<a href="https://colab.research.google.com/github/romashko1977/skills-github-pages/blob/main/bot_qwantum_istiry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ccxt pandas_ta plotly

In [ ]:
import ccxt, time, pandas as pd, pandas_ta as ta, numpy as np, requests
from datetime import datetime
from IPython.display import display, HTML, clear_output
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

# =====================================================================
# ⚙️ КОНФИГУРАЦИЯ
# =====================================================================
class Config:
    API_KEY = 'mx0vglRyG4Ee1fFsfV'
    API_SECRET = '8c1929b7d096420dadbc416f6fd6035d'

    Z_THRESHOLD = 2.7
    GANN_WINDOWS = [90, 144, 270]
    RR_RATIO = 2.6
    SL_ATR_MULT = 1.7
    SCAN_INTERVAL = 30
    BACKTEST_CANDLES = 600 # Глубина теста (свечи 15м)

# =====================================================================
# 📊 МОДУЛЬ АВТО-ТЕСТА
# =====================================================================
def run_quick_backtest(exchange, symbol, current_df):
    """Автоматический тест стратегии на истории конкретной монеты"""
    df = current_df.copy()
    # Дозагружаем историю для теста, если её мало в основном сканере
    if len(df) < Config.BACKTEST_CANDLES:
        ohlcv = exchange.fetch_ohlcv(symbol, '15m', limit=Config.BACKTEST_CANDLES)
        df = pd.DataFrame(ohlcv, columns=['ts','o','h','l','c','v'])

    df['dt'] = pd.to_datetime(df['ts'], unit='ms')
    ema = df['c'].ewm(span=20).mean()
    std = df['c'].rolling(20).std()
    df['z'] = (df['c'] - ema) / std
    df['atr'] = ta.atr(df['h'], df['l'], df['c'], length=14)

    wins, losses = 0, 0
    for i in range(25, len(df) - 24):
        row = df.iloc[i]
        mins = row['dt'].hour * 60 + row['dt'].minute
        if abs(row['z']) > Config.Z_THRESHOLD and any(mins % v < 15 for v in Config.GANN_WINDOWS):
            side = 'SHORT' if row['z'] > 0 else 'LONG'
            sl = row['c'] + (row['atr']*1.7) if side=='SHORT' else row['c'] - (row['atr']*1.7)
            tp = row['c'] - (row['atr']*1.7*2.6) if side=='SHORT' else row['c'] + (row['atr']*1.7*2.6)

            for j in range(i+1, i+25):
                if (side=='LONG' and df.iloc[j]['l'] <= sl) or (side=='SHORT' and df.iloc[j]['h'] >= sl):
                    losses += 1; break
                if (side=='LONG' and df.iloc[j]['h'] >= tp) or (side=='SHORT' and df.iloc[j]['l'] <= tp):
                    wins += 1; break

    total = wins + losses
    wr = (wins / total * 100) if total > 0 else 0
    pf = (wins * 2.6) / losses if losses > 0 else wins * 2.6
    return {'wr': wr, 'pf': pf, 'trades': total}

# =====================================================================
# 🧠 ОСНОВНОЙ ДВИЖОК
# =====================================================================
class QuantumAutoStation:
    def __init__(self):
        self.mexc = ccxt.mexc({'apiKey': Config.API_KEY, 'secret': Config.API_SECRET, 'enableRateLimit': True, 'options': {'defaultType': 'swap'}})

    def get_data(self):
        self.mexc.load_markets()
        symbols = [s for s in self.mexc.symbols if '/USDT:' in s][:60]
        now_m = datetime.utcnow().hour * 60 + datetime.utcnow().minute
        vibe = any(now_m % v < 10 for v in Config.GANN_WINDOWS)

        results = []
        for s in symbols:
            try:
                ohlcv = self.mexc.fetch_ohlcv(s, '15m', limit=100)
                df = pd.DataFrame(ohlcv, columns=['ts','o','h','l','c','v'])
                z = ((df['c'] - df['c'].ewm(20).mean()) / df['c'].rolling(20).std()).iloc[-1]
                acc = (df['c'].iloc[-1] - df['c'].iloc[-4]) / df['c'].iloc[-4] * 100
                score = abs(z) * 2 + (abs(acc) * 0.5) + (4 if vibe else 0)

                results.append({'coin': s, 'z': z, 'acc': acc, 'score': score, 'price': df['c'].iloc[-1], 'df': df})
            except: continue
        return sorted(results, key=lambda x: x['score'], reverse=True), vibe

# =====================================================================
# 🚀 ЗАПУСК
# =====================================================================
station = QuantumAutoStation()

while True:
    try:
        data, is_vibe = station.get_data()
        clear_output(wait=True)

        # Отрисовка таблицы
        rows = ""
        for c in data[:12]:
            col = "#ff4d4d" if c['z'] > 0 else "#00ff88"
            rows += f"<tr style='border-bottom:1px solid #222;'><td style='padding:8px;'><b>{c['coin'].split('/')[0]}</b></td><td style='color:{col};'>{c['z']:+.2f}σ</td><td style='color:#00f3ff;'>{c['price']:.5f}</td><td style='text-align:center;'><b>{c['score']:.1f}</b></td></tr>"

        display(HTML(f"<div style='background:#0b0e11; padding:15px; border-radius:10px; font-family:monospace; color:#848e9c;'><div style='display:flex; justify-content:space-between; color:#00f3ff;'><b>ALPHA v19.0 AUTO-TEST</b><b>{'VIBRATION 🌀' if is_vibe else ''}</b></div><table style='width:100%; text-align:left;'><tr><th>ASSET</th><th>Z-SC</th><th>PRICE</th><th>SCORE</th></tr>{rows}</table></div>"))

        # АВТО-ТЕСТ ДЛЯ ТОП-ПРЕТЕНДЕНТА
        if data and data[0]['score'] >= 8.5:
            top = data[0]
            test = run_quick_backtest(station.mexc, top['coin'], top['df'])

            # Вывод результата теста под таблицей
            res_col = "#00ff88" if test['wr'] >= 35 else "#ff4d4d"
            display(HTML(f"""
            <div style='background:#161a1e; margin-top:10px; padding:15px; border-radius:10px; border: 1px solid {res_col}; font-family:monospace;'>
                <b style='color:#00f3ff;'>HISTORICAL VALIDATION: {top['coin'].split('/')[0]}</b><br>
                <span style='color:{res_col};'>WinRate: {test['wr']:.1f}%</span> |
                <span>Profit Factor: {test['pf']:.2f}</span> |
                <span>Samples: {test['trades']} signals</span>
            </div>
            """))

            # График
            df_g = top['df']
            fig = go.Figure(data=[go.Candlestick(x=pd.to_datetime(df_g['ts'], unit='ms'), open=df_g['o'], high=df_g['h'], low=df_g['l'], close=df_g['c'])])
            fig.update_layout(title=f"LIVE: {top['coin']} (Prob: {top['score']:.1f})", template="plotly_dark", height=350, xaxis_rangeslider_visible=False, margin=dict(t=30,b=10))
            fig.show()

    except Exception as e: print(f"Error: {e}")
    time.sleep(Config.SCAN_INTERVAL)

ASSET,Z-SC,PRICE,SCORE
ALUMINUM,+1286572.68σ,3519.34000,2573149.4
4,-4.77σ,0.01170,14.3
ALLO,+3.22σ,0.10918,10.6
APR,+3.14σ,0.18189,10.4
AIOT,+2.76σ,0.01421,10.3
APEX,-2.95σ,0.27220,10.1
AR,-2.61σ,1.72500,9.8
ALCH,-2.51σ,0.06010,9.7
1000000BABYDOGE,-2.38σ,0.00039,9.3
AAVE,-2.40σ,98.26000,9.2
